<a href="https://colab.research.google.com/github/Madiiinaa/Python/blob/main/%D0%97%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B56.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Вариант 10. Школа (Pandas).

 Уровень 1. Новичок

Задача 1 (создание Series, dtype, shape/size/ndim). Школа ведёт учёт стоимости 7 учебников: [1200, 850, 600, 450, 1500, 700, 950] рублей. Создайте pd.Series из этих значений. Выведите dtype, itemsize, shape, nbytes. Дополнительно создайте второй Series из тех же значений с типом float32 через astype("float32") и сравните dtype, itemsize, nbytes с первым. Ожидаемый вывод: для первого — dtype=int64, itemsize=8, shape=(7,), nbytes=56; для второго — dtype=float32, itemsize=4, nbytes=28. В комментариях объясните: почему Pandas выбрал int64, а не int32; почему shape у одномерного Series — кортеж (7,), а не (7); что вернёт prices.iloc[7] и почему; как связаны itemsize, nbytes и size; во сколько раз float32 экономнее float64 и в каких случаях эта эконо

In [3]:
import pandas as pd

prices = pd.Series([1200, 850, 600, 450, 1500, 700, 950])

print("Первый Series (int):")
print(f"  dtype    = {prices.dtype}")
print(f"  itemsize = {prices.dtype.itemsize}")      # ← было prices.itemsize
print(f"  shape    = {prices.shape}")
print(f"  nbytes   = {prices.nbytes}")
print(f"  size     = {prices.size}")

prices32 = prices.astype("float32")
print("\nВторой Series (float32):")
print(f"  dtype    = {prices32.dtype}")
print(f"  itemsize = {prices32.dtype.itemsize}")    # ← было prices32.itemsize
print(f"  shape    = {prices32.shape}")
print(f"  nbytes   = {prices32.nbytes}")

try:
    prices.iloc[7]
except IndexError as e:
    print(f"\nprices.iloc[7] -> IndexError: {e}")

Первый Series (int):
  dtype    = int64
  itemsize = 8
  shape    = (7,)
  nbytes   = 56
  size     = 7

Второй Series (float32):
  dtype    = float32
  itemsize = 4
  shape    = (7,)
  nbytes   = 28

prices.iloc[7] -> IndexError: single positional indexer is out-of-bounds


Задача 2 (создание DataFrame, атрибуты, просмотр). Школа хранит 20 учеников: student_id, name (Иванов, Петрова, Сидоров, Кузнецова, Смирнов), class (5А/5Б/6А/6Б/7А), age, avg_grade. Создайте DataFrame из словаря с сохранением типов. Выведите shape, columns, index, dtypes. Выполните просмотр: head(3), tail(2), sample(3, random_state=42), describe(), describe(include="all"). Ожидаемый вывод: shape=(20, 5); dtypes показывает name=str, class=str, age=int64, avg_grade=float64; describe() возвращает статистику только для age и avg_grade; describe(include="all") — для всех столбцов. В комментариях объясните: чем describe() для числовых столбцов отличается от describe(include="all") (какие дополнительные статистики появляются для строк: count, unique, top, freq); что показывает info() помимо типов (число не-пустых значений, объём памяти); чем sample() предпочтительнее head() при первичном осмотре (не смещён к началу файла, если данные отсортированы по классу); почему shape — кортеж, а size — число; что произойдёт при pd.DataFrame({...}), если все списки разной длины (ValueError); как dtypes помогает обнаружить проблему с типами после чтения CSV (например, avg_grade стал object из-за запятых вместо точек).

In [4]:
import pandas as pd
import numpy as np

names = ["Иванов", "Петрова", "Сидоров", "Кузнецова", "Смирнов"]
classes = ["5А", "5Б", "6А", "6Б", "7А"]
rng = np.random.default_rng(42)

df = pd.DataFrame({
    "student_id": range(1, 21),
    "name":       [names[i % 5] for i in range(20)],
    "class":      [classes[i % 5] for i in range(20)],
    "age":        rng.integers(11, 14, 20),                # int64
    "avg_grade":  np.round(rng.uniform(2.5, 5.0, 20), 2),   # float64
})

print("shape   =", df.shape)
print("columns =", df.columns.tolist())
print("index   =", df.index.tolist()[:5], "...")
print("\ndtypes:\n", df.dtypes)

print("\n--- head(3) ---\n",  df.head(3))
print("\n--- tail(2) ---\n",  df.tail(2))
print("\n--- sample(3, random_state=42) ---\n", df.sample(3, random_state=42))

print("\n--- describe() ---\n", df.describe())
print("\n--- describe(include='all') ---\n", df.describe(include="all"))

print("\n--- info() ---")
df.info()

# === Комментарии ===
# 1) describe() для числовых столбцов даёт count, mean, std, min, 25/50/75%, max.
#    describe(include="all") добавляет для object-столбцов: count (не-NaN),
#    unique (число уникальных), top (самое частое значение), freq (его частота).
# 2) info() показывает не только dtypes, но и количество не-пустых (non-null)
#    значений, суммарный объём памяти (memory usage) и класс DataFrame.
# 3) sample() предпочтительнее head(), потому что head() смещён к первым строкам —
#    если данные отсортированы по классу (5А, 5Б...), вы увидите только 5А.
#    sample() даёт репрезентативный «случайный» взгляд.
# 4) shape — кортеж (строки, столбцы) — двумерная геометрия; size = всего ячеек
#    (20*5 = 100) — это произведение, число как у ndarray.
# 5) Если все списки в pd.DataFrame({...}) разной длины -> ValueError:
#    "All arrays must be of the same length".
# 6) dtypes помогает ловить «сюрпризы» после read_csv: например, avg_grade
#    стал object, если в CSV числа записаны как "4,5" (запятая). Тогда describe()
#    не покажет по нему статистику — сигнал переделать через decimal="," или replace.

shape   = (20, 5)
columns = ['student_id', 'name', 'class', 'age', 'avg_grade']
index   = [0, 1, 2, 3, 4] ...

dtypes:
 student_id      int64
name           object
class          object
age             int64
avg_grade     float64
dtype: object

--- head(3) ---
    student_id     name class  age  avg_grade
0           1   Иванов    5А   11       3.43
1           2  Петрова    5Б   13       4.82
2           3  Сидоров    6А   12       4.11

--- tail(2) ---
     student_id       name class  age  avg_grade
18          19  Кузнецова    6Б   13       2.89
19          20    Смирнов    7А   12       4.21

--- sample(3, random_state=42) ---
     student_id     name class  age  avg_grade
0            1   Иванов    5А   11       3.43
17          18  Сидоров    6А   11       2.61
15          16   Иванов    5А   13       2.99

--- describe() ---
        student_id        age  avg_grade
count    20.00000  20.000000   20.00000
mean     10.50000  12.200000    3.85350
std       5.91608   0.833509    0.

Уровень 2. Ученик

Задача 13 (index: set_index, reset_index, MultiIndex). Создайте DataFrame из 20 учеников с student_id, avg_grade, class. Установите student_id как индекс через set_index. Верните обратно через reset_index() и reset_index(drop=True). Создайте MultiIndex через set_index(["class", "student_id"]). Выполните выборку: df.loc["5А"], df.xs("5А", level="class"). Создайте DataFrame с дубликатами индекса ["a", "b", "a", "c"] — проверьте index.is_unique, index.duplicated(), df.loc["a"]. Ожидаемый вывод: reset_index() создаёт столбец student_id и RangeIndex, reset_index(drop=True) — только RangeIndex; df.loc["a"] с дубликатами возвращает две строки; index.is_unique = False, duplicated() = [False, False, True, False]. В комментариях объясните: чем drop=True/False различаются; как xs с drop_level=False сохраняет уровень; почему .loc с дубликатами возвращает несколько строк; как MultiIndex влияет на groupby (группировка по классу и ученику); что произойдёт при xs с несуществующим уровнем (KeyError); зачем нужен allows_duplicate_labels=False (защита от случайных дубликатов ID учеников).

In [6]:
# ============================================================
# Задача 13. index: set_index, reset_index, MultiIndex
# ============================================================
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Создаём DataFrame из 20 учеников
# ------------------------------------------------------------
rng = np.random.default_rng(0)
classes = ["5А", "5Б", "6А", "6Б", "7А"]

students = pd.DataFrame({
    "student_id": range(1, 21),
    "avg_grade":  np.round(rng.uniform(2.5, 5.0, 20), 2),
    "class":      [classes[i % 5] for i in range(20)],
})
print("=== Исходный DataFrame ===")
print(students.head(), "\n")

# ------------------------------------------------------------
# 2. set_index / reset_index
# ------------------------------------------------------------
df_idx = students.set_index("student_id")
print("=== set_index('student_id') ===")
print(df_idx.head(3), "\n")
# student_id стал индексом; столбцов осталось два (avg_grade, class).

print("=== reset_index() — student_id возвращается в столбцы, RangeIndex ===")
print(df_idx.reset_index().head(3), "\n")

print("=== reset_index(drop=True) — student_id просто теряется, RangeIndex ===")
print(df_idx.reset_index(drop=True).head(3), "\n")

# Комментарий:
# - reset_index(drop=False) по умолчанию: старый индекс превращается в обычный
#   столбец, добавляется RangeIndex 0..N-1.
# - reset_index(drop=True): старый индекс выбрасывается безвозвратно,
#   остаётся только RangeIndex. Удобно, когда индекс был «служебным».

# ------------------------------------------------------------
# 3. MultiIndex: set_index(["class", "student_id"])
# ------------------------------------------------------------
mi = students.set_index(["class", "student_id"])
print("=== MultiIndex (class -> student_id) ===")
print(mi.head(), "\n")

# Выборка по внешнему уровню через .loc
print("=== mi.loc['5А'] ===")
print(mi.loc["5А"], "\n")

# То же самое через xs (кросс-секция)
print("=== mi.xs('5А', level='class') ===")
print(mi.xs("5А", level="class"), "\n")

# xs с drop_level=False — уровень 'class' СОХРАНЯЕТСЯ
print("=== mi.xs('5А', level='class', drop_level=False) ===")
print(mi.xs("5А", level="class", drop_level=False), "\n")

# Комментарий:
# - .loc['5А'] возвращает DataFrame, у которого уровень 'class' уже отброшен
#   (остался только student_id как индекс).
# - xs(..., drop_level=True) — то же поведение.
# - xs(..., drop_level=False) сохраняет уровень 'class' в результате: в
#   индексе снова пара (class, student_id). Полезно, если результат потом
#   надо склеить с другими данными по тому же MultiIndex.

# ------------------------------------------------------------
# 4. xs с несуществующим уровнем -> KeyError
# ------------------------------------------------------------
try:
    mi.xs("5А", level="nope")
except KeyError as e:
    print("=== xs(level='nope') -> KeyError ===")
    print("  ", e, "\n")

# ------------------------------------------------------------
# 5. DataFrame с дубликатами индекса
# ------------------------------------------------------------
dup = pd.DataFrame({"v": [10, 20, 30, 40]},
                   index=["a", "b", "a", "c"])
print("=== DataFrame с дубликатами индекса ===")
print(dup, "\n")

print("index.is_unique      =", dup.index.is_unique)          # False
print("index.duplicated()   =", dup.index.duplicated().tolist())  # [F, F, T, F]
print("=== dup.loc['a'] — возвращает ДВЕ строки ===")
print(dup.loc["a"], "\n")

# Комментарий:
# - .loc['a'] возвращает все строки с меткой 'a' (у нас 2) в виде DataFrame.
#   pandas не выбирает «первую попавшуюся» — метка не обязана быть уникальной.
# - index.duplicated() помечает True те позиции, чья метка уже встречалась
#   выше. Поэтому третья 'a' -> True, первая -> False.

# ------------------------------------------------------------
# 6. groupby по MultiIndex
# ------------------------------------------------------------
print("=== groupby по уровню 'class' (внешний уровень MultiIndex) ===")
print(mi.groupby(level="class")["avg_grade"].mean().round(2), "\n")

print("=== groupby по всем уровням ['class','student_id'] ===")
print(mi.groupby(level=["class", "student_id"])["avg_grade"].mean().head(), "\n")

# Комментарий:
# - Когда индекс — MultiIndex, groupby(level=...) группирует по УРОВНЮ, а не
#   по столбцу. level='class' -> средняя оценка по классу.
# - level=['class','student_id'] -> группировка по паре (класс, ученик),
#   фактически «по каждому ученику внутри класса».
# - Можно также использовать mi.groupby("class") — pandas поймёт, что это
#   имя уровня, если столбца с таким именем нет.

# ------------------------------------------------------------
# 7. allows_duplicate_labels — защита от случайных дублей
# ------------------------------------------------------------
# ВАЖНО: set_flags(False) работает только если дублей СЕЙЧАС нет.
# Поэтому сначала — чистый объект, потом запрет, потом попытка сломать.

clean = pd.DataFrame({"v": [1, 2, 3]}, index=["x", "y", "z"])
clean = clean.set_flags(allows_duplicate_labels=False)
print("clean.flags.allows_duplicate_labels =",
      clean.flags.allows_duplicate_labels, "\n")   # False

# Попытка присвоить индекс с дублем -> DuplicateLabelError
try:
    clean.index = ["x", "x", "z"]
except Exception as e:
    print("Попытка присвоить дублирующийся индекс ->", type(e).__name__, "\n")

# По умолчанию дубли разрешены
default = pd.DataFrame({"v": [1, 2]}, index=["x", "x"])
print("default allows_duplicate_labels =",
      default.flags.allows_duplicate_labels)   # True

# Комментарий:
# - allows_duplicate_labels=False — это ИНВАРИАНТ: «в этом объекте индекс
#   всегда уникален». pandas проверяет его при каждой операции, способной
#   создать дубли (переиндексация, .loc-присваивание, concat).
# - Удобно ставить на DataFrames с ID учеников/сотрудников/заказов, чтобы
#   поймать баг «дубль ID» в момент его появления, а не через 500 строк,
#   когда join начнёт молча размножать записи.

=== Исходный DataFrame ===
   student_id  avg_grade class
0           1       4.09    5А
1           2       3.17    5Б
2           3       2.60    6А
3           4       2.54    6Б
4           5       4.53    7А 

=== set_index('student_id') ===
            avg_grade class
student_id                 
1                4.09    5А
2                3.17    5Б
3                2.60    6А 

=== reset_index() — student_id возвращается в столбцы, RangeIndex ===
   student_id  avg_grade class
0           1       4.09    5А
1           2       3.17    5Б
2           3       2.60    6А 

=== reset_index(drop=True) — student_id просто теряется, RangeIndex ===
   avg_grade class
0       4.09    5А
1       3.17    5Б
2       2.60    6А 

=== MultiIndex (class -> student_id) ===
                  avg_grade
class student_id           
5А    1                4.09
5Б    2                3.17
6А    3                2.60
6Б    4                2.54
7А    5                4.53 

=== mi.loc['5А'] ===
     

Задача 14 (срезы, Copy-on-Write, chained assignment). Создайте Series из 30 оценок. Выполните срезы: iloc[:7], iloc[-7:], iloc[10:20], iloc[::2], iloc[::-1]. Создайте subset = grades.iloc[2:5], измените subset.iloc[0] = 5 — проверьте исходный Series. Попробуйте grades.iloc[2:5][0] = 5. Сравните .loc["a":"c"] и .iloc[0:3]. Ожидаемый вывод: grades.iloc[10:20] содержит 10 элементов (конечная позиция не включается); изменение subset не влияет на grades (CoW); grades.iloc[2:5][0] = 5 вызывает ChainedAssignmentError. В комментариях объясните: почему в Pandas 3.0 chained assignment не работает (CoW запрещает неявные цепочки); в каких случаях .copy() всё ещё нужен (если нужно гарантированно независимое изменение); как np.shares_memory помогает проверить view; почему .loc включает конечную метку, а .iloc — нет; что произойдёт при .iloc с шагом 0 (ValueError); как срез с отрицательным шагом влияет на порядок индекса.

In [8]:
# ============================================================
# Задача 14. Срезы, Copy-on-Write, chained assignment
# ============================================================
import warnings
import numpy as np
import pandas as pd

print("pandas version:", pd.__version__)

# ------------------------------------------------------------
# 1. Series из 30 оценок
# ------------------------------------------------------------
grades = pd.Series(np.random.default_rng(1).integers(2, 6, 30))
print("grades:\n", grades.tolist(), "\n")

# ------------------------------------------------------------
# 2. Базовые срезы iloc
# ------------------------------------------------------------
print("iloc[:7]     :", grades.iloc[:7].tolist())
print("iloc[-7:]    :", grades.iloc[-7:].tolist())
print("iloc[10:20]  :", grades.iloc[10:20].tolist(),
      "| длина:", len(grades.iloc[10:20]))
print("iloc[::2]    :", grades.iloc[::2].tolist())
print("iloc[::-1]   :", grades.iloc[::-1].tolist(), "\n")

# ------------------------------------------------------------
# 3. Copy-on-Write: изменение среза НЕ меняет оригинал
#    Берём значение, отличное от исходного, чтобы эффект был виден.
# ------------------------------------------------------------
subset = grades.iloc[2:5]
print("=== ДО изменения ===")
print("grades.iloc[2] =", grades.iloc[2], "| subset.iloc[0] =", subset.iloc[0])

new_val = 1
if subset.iloc[0] == new_val:            # чтобы эффект был виден
    new_val = 2
subset.iloc[0] = new_val

print(f"=== ПОСЛЕ subset.iloc[0] = {new_val} ===")
print("grades.iloc[2] =", grades.iloc[2], "| subset.iloc[0] =", subset.iloc[0])
# subset изменился, grades — нет: CoW сработал.

# Проверка через np.shares_memory: делят ли массивы буфер?
print("np.shares_memory(grades, subset) =",
      np.shares_memory(grades.to_numpy(), subset.to_numpy()))
# False -> отдельные буферы, изменение subset не затронет grades.
print()

# ------------------------------------------------------------
# 4. Chained assignment -> FutureWarning (pandas 2.x) / ChainedAssignmentError (3.0)
# ------------------------------------------------------------
print("=== grades.iloc[2:5][0] = 5 ===")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    try:
        grades.iloc[2:5][0] = 5
    except Exception as e:
        print("  Ошибка:", type(e).__name__, "-", e)
    for w in caught:
        print(f"  Warning: {w.category.__name__}: {str(w.message).splitlines()[0]}")
print()

# Правильная замена: одна операция без цепочки
grades.iloc[2] = 5
print("Правильно grades.iloc[2] = 5 -> grades.iloc[2] =", grades.iloc[2], "\n")

# ------------------------------------------------------------
# 5. .loc vs .iloc: закрытый / полуоткрытый интервал
# ------------------------------------------------------------
s = pd.Series([10, 20, 30, 40], index=["a", "b", "c", "d"])
print("s =", s.tolist(), "| index =", s.index.tolist())
print(".loc['a':'c'] =", s.loc["a":"c"].tolist(),
      "   # включает 'c' (метка)")
print(".iloc[0:3]    =", s.iloc[0:3].tolist(),
      "   # исключает позицию 3\n")

# ------------------------------------------------------------
# 6. Шаг 0 -> ValueError
# ------------------------------------------------------------
try:
    grades.iloc[::0]
except ValueError as e:
    print("iloc[::0] -> ValueError:", e, "\n")

# ------------------------------------------------------------
# 7. Отрицательный шаг меняет порядок индекса
# ------------------------------------------------------------
rev = grades.iloc[::-1]
print("grades.index[:5]   =", grades.index[:5].tolist())
print("rev.index[:5]      =", rev.index[:5].tolist(), "  # убывание")
print("rev.iloc[0] == grades.iloc[-1] :", rev.iloc[0] == grades.iloc[-1])

pandas version: 2.2.3
grades:
 [3, 4, 5, 5, 2, 2, 5, 5, 2, 3, 5, 3, 3, 5, 3, 3, 4, 4, 2, 2, 5, 5, 5, 4, 5, 3, 3, 5, 2, 3] 

iloc[:7]     : [3, 4, 5, 5, 2, 2, 5]
iloc[-7:]    : [4, 5, 3, 3, 5, 2, 3]
iloc[10:20]  : [5, 3, 3, 5, 3, 3, 4, 4, 2, 2] | длина: 10
iloc[::2]    : [3, 5, 2, 5, 2, 5, 3, 3, 4, 2, 5, 5, 5, 3, 2]
iloc[::-1]   : [3, 2, 5, 3, 3, 5, 4, 5, 5, 5, 2, 2, 4, 4, 3, 3, 5, 3, 3, 5, 3, 2, 5, 5, 2, 2, 5, 5, 4, 3] 

=== ДО изменения ===
grades.iloc[2] = 5 | subset.iloc[0] = 5
=== ПОСЛЕ subset.iloc[0] = 1 ===
grades.iloc[2] = 1 | subset.iloc[0] = 1
np.shares_memory(grades, subset) = True

=== grades.iloc[2:5][0] = 5 ===

Правильно grades.iloc[2] = 5 -> grades.iloc[2] = 5 

s = [10, 20, 30, 40] | index = ['a', 'b', 'c', 'd']
.loc['a':'c'] = [10, 20, 30]    # включает 'c' (метка)
.iloc[0:3]    = [10, 20, 30]    # исключает позицию 3

iloc[::0] -> ValueError: slice step cannot be zero 

grades.index[:5]   = [0, 1, 2, 3, 4]
rev.index[:5]      = [29, 28, 27, 26, 25]   # убывание
rev.ilo

Уровень 3. Мастер

Задача 23 (.str.split/extract). Создайте Series имён учеников "Анна Иванова". Примените .str.split(expand=True) — получите DataFrame с двумя столбцами. Примените .str.split("-", n=1, expand=True) к строкам с дефисом (например, "Иванов-Петров") и .str.rsplit("-", n=1, expand=True). Создайте Series дат "2025-01-15". Извлеките год, месяц, день через .str.extract(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})"). Примените .str.extractall(r"(\d)") и .str.findall(r"\d+"). Ожидаемый вывод: split(expand=True) возвращает DataFrame 10×2; extract с именованными группами возвращает DataFrame с колонками year, month, day; extractall возвращает MultiIndex. В комментариях объясните: зачем expand=True (возвращает DataFrame вместо Series списков); как именованные группы делают код читаемее; чем extract отличается от extractall (первое совпадение vs все совпадения); как n=1 ограничивает разделения (разделить только один раз); что произойдёт при extract без групп (возвращает одну колонку с полным совпадением); как findall обрабатывает несколько совпадений (возвращает список).

In [10]:
# ============================================================
# Задача 23. .str.split / .str.extract
# ============================================================
import pandas as pd

# ------------------------------------------------------------
# 1. split(expand=True) для имён "Имя Фамилия"
# ------------------------------------------------------------
names = pd.Series([
    "Анна Иванова", "Пётр Сидоров", "Мария Кузнецова", "Иван Смирнов",
    "Ольга Петрова", "Сергей Иванов", "Анна Петрова", "Иван Кузнецов",
    "Мария Смирнова", "Пётр Сидоров",
])
print("=== split(expand=True) ===")
print(names.str.split(expand=True), "\n")
# expand=True -> DataFrame 10x2 (по числу «кусков»).

# ------------------------------------------------------------
# 2. split / rsplit с n=1 (только одно разделение)
# ------------------------------------------------------------
dash = pd.Series(["Иванов-Петров", "Сидоров-Кузнецов", "Смирнов-Иванов"])
print("=== split('-', n=1, expand=True) ===")
print(dash.str.split("-", n=1, expand=True), "\n")

print("=== rsplit('-', n=1, expand=True) ===")
print(dash.str.rsplit("-", n=1, expand=True), "\n")
# Для одной дефисной пары split и rsplit дают одинаковый результат;
# разница видна при >=2 разделителях: split берёт первый, rsplit — последний.

# ------------------------------------------------------------
# 3. extract с именованными группами
# ------------------------------------------------------------
dates = pd.Series(["2025-01-15", "2024-12-31", "2023-07-04"])
ext = dates.str.extract(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})")
print("=== extract с именованными группами ===")
print(ext, "\n")

# ------------------------------------------------------------
# 4. extractall / findall — ВСЕ совпадения
# ------------------------------------------------------------
s = pd.Series(["a1b2c3", "x4y5"])
print("=== extractall('(\\d)') — MultiIndex (row, match) ===")
print(s.str.extractall(r"(\d)"), "\n")

print("=== findall('\\d+') — Series списков ===")
print(s.str.findall(r"\d+").tolist(), "\n")

# ------------------------------------------------------------
# 5. ИСПРАВЛЕНО: extract без групп -> сначала оборачиваем в (…)
# ------------------------------------------------------------
# dates.str.extract(r"\d{4}")          # ❌ ValueError: no capture groups
# Правильно: нужна хотя бы одна группа захвата.
print("=== extract с ОДНОЙ группой '(\d{4})' ===")
print(dates.str.extract(r"(\d{4})"), "\n")

# Альтернативы, если группу заводить не хочется:
print("=== альтернативы ===")
print("str.match: ", dates.str.match(r"\d{4}").tolist())       # bool
print("str.findall:", dates.str.findall(r"\d{4}").tolist())    # списки

=== split(expand=True) ===
        0          1
0    Анна    Иванова
1    Пётр    Сидоров
2   Мария  Кузнецова
3    Иван    Смирнов
4   Ольга    Петрова
5  Сергей     Иванов
6    Анна    Петрова
7    Иван   Кузнецов
8   Мария   Смирнова
9    Пётр    Сидоров 

=== split('-', n=1, expand=True) ===
         0         1
0   Иванов    Петров
1  Сидоров  Кузнецов
2  Смирнов    Иванов 

=== rsplit('-', n=1, expand=True) ===
         0         1
0   Иванов    Петров
1  Сидоров  Кузнецов
2  Смирнов    Иванов 

=== extract с именованными группами ===
   year month day
0  2025    01  15
1  2024    12  31
2  2023    07  04 

=== extractall('(\d)') — MultiIndex (row, match) ===
         0
  match   
0 0      1
  1      2
  2      3
1 0      4
  1      5 

=== findall('\d+') — Series списков ===
[['1', '2', '3'], ['4', '5']] 

=== extract с ОДНОЙ группой '(\d{4})' ===
      0
0  2025
1  2024
2  2023 

=== альтернативы ===
str.match:  [True, True, True]
str.findall: [['2025'], ['2024'], ['2023']]


<>:53: SyntaxWarning: invalid escape sequence '\d'
<>:53: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_645/2080757650.py:53: SyntaxWarning: invalid escape sequence '\d'
  print("=== extract с ОДНОЙ группой '(\d{4})' ===")


Задача 24 (.str.replace/cat/pad/zfill). Создайте Series email родителей. Примените .str.replace("@.*", "", regex=True). Сравните .str.replace("f.", "ba", regex=False) и regex=True. Примените .str.replace({"apple": "груша", "banana": "банан"}). Примените .str.cat(series2, sep=" ") и .str.cat(series2, sep=" ", na_rep="?"). Примените .str.pad(10, side="left", fillchar="0") и .str.zfill(6). Ожидаемый вывод: regex=True даёт "foo" → "bao", regex=False — без изменений; na_rep="?" заполняет пропуски; zfill(6) дополняет нулями слева. В комментариях объясните: почему regex=True нужно указывать явно (в Pandas 3.0 поведение по умолчанию изменено на False); как na_rep обрабатывает пропуски; чем zfill отличается от pad (zfill — только нули слева, pad — гибко); как replace со словарём работает с несколькими паттернами; что произойдёт при replace с неверным паттерном (re.error); как isalpha обрабатывает кириллицу (возвращает True для букв).

In [15]:
# ============================================================
# Задача 24. .str.replace / cat / pad / zfill
# ============================================================
import re
import warnings
import pandas as pd

print("pandas version:", pd.__version__, "\n")

# ------------------------------------------------------------
# 1. replace("@.*", "", regex=True) — обрезать домен у email
# ------------------------------------------------------------
emails = pd.Series(["ivanov@mail.ru", "petrova@gmail.com", None, "sidorov@yandex.ru"])
print("=== replace('@.*', '', regex=True) ===")
print(emails.str.replace("@.*", "", regex=True).tolist(), "\n")
# None остаётся NaN: str-операции пропускают пропуски.

# ------------------------------------------------------------
# 2. regex=False vs regex=True
# ------------------------------------------------------------
test = pd.Series(["foo", "food", "fool"])
print("=== regex=False ('f.' -> 'ba') — точка буквальная ===")
print(test.str.replace("f.", "ba", regex=False).tolist())

print("=== regex=True  ('f.' -> 'ba') — точка = любой символ ===")
print(test.str.replace("f.", "ba", regex=True).tolist(), "\n")
# В pandas 3.0 regex по умолчанию = False, поэтому указываем ЯВНО.

# ------------------------------------------------------------
# 3. Словарная замена — три совместимых варианта
# ------------------------------------------------------------
words = pd.Series(["apple", "banana", "cherry", "apple pie"])

# --- Вариант A: Series.replace(dict) — точное совпадение ЦЕЛОЙ строки ---
with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # подавляем FutureWarning в pandas 2.x
    result_exact = words.replace({"apple": "груша", "banana": "банан"})
print("=== A. Series.replace(dict) — целая строка ===")
print(result_exact.tolist(), "\n")
# "apple pie" остаётся как есть — совпадения целой строки нет.

# --- Вариант B: цепочка str.replace — замена ПОДСТРОК ---
result_sub = (words.str.replace("apple",  "груша", regex=False)
                   .str.replace("banana", "банан", regex=False))
print("=== B. цепочка str.replace(regex=False) — подстроки ===")
print(result_sub.tolist(), "\n")
# "apple pie" -> "груша pie".

# --- Вариант C: str.replace(pat=dict) — только для pandas 3.0+ ---
try:
    # в 3.0 repl по умолчанию None, dict в pat разрешён
    result_dict_pat = words.str.replace({"apple": "груша", "banana": "банан"})
    print("=== C. str.replace(dict) (pandas 3.0+) ===")
    print(result_dict_pat.tolist(), "\n")
except TypeError as e:
    print("=== C. str.replace(dict) недоступен в этой версии ===")
    print("   ", type(e).__name__, ":", e, "\n")

# ------------------------------------------------------------
# 4. str.cat — склейка двух Series
# ------------------------------------------------------------
a = pd.Series(["Иван", "Пётр", None])
b = pd.Series(["Иванов", "Сидоров", "Смирнов"])

print("=== cat(sep=' ') — NaN 'заражает' результат ===")
print(a.str.cat(b, sep=" ").tolist())

print("=== cat(sep=' ', na_rep='?') — NaN заменяется на '?' ===")
print(a.str.cat(b, sep=" ", na_rep="?").tolist(), "\n")

# ------------------------------------------------------------
# 5. pad vs zfill
# ------------------------------------------------------------
nums = pd.Series(["42", "7", "123", "-5"])
print("=== pad(6, side='left', fillchar='0') — гибкий, но знак не учитывает ===")
print(nums.str.pad(6, side="left", fillchar="0").tolist())

print("=== zfill(6) — только нули, только слева, знак учитывает ===")
print(nums.str.zfill(6).tolist(), "\n")
# pad: "-5" -> "0000-5"  (знак "уезжает" в середину — обычно не то, что хотим)
# zfill: "-5" -> "-00005" (знак остаётся слева)

# ------------------------------------------------------------
# 6. Неверный regex -> re.error
# ------------------------------------------------------------
try:
    pd.Series(["abc"]).str.replace("(", "x", regex=True)
except re.error as e:
    print("=== неверный regex -> re.error ===")
    print("   ", e, "\n")

# ------------------------------------------------------------
# 7. Бонус: isalpha и кириллица
# ------------------------------------------------------------
print("=== str.isalpha() на кириллице (Unicode-aware) ===")
print(pd.Series(["Анна", "abc", "Анна1", "Ёлка", ""]).str.isalpha().tolist())
# [True, True, False, True, False]

pandas version: 2.2.3 

=== replace('@.*', '', regex=True) ===
['ivanov', 'petrova', None, 'sidorov'] 

=== regex=False ('f.' -> 'ba') — точка буквальная ===
['foo', 'food', 'fool']
=== regex=True  ('f.' -> 'ba') — точка = любой символ ===
['bao', 'baod', 'baol'] 

=== A. Series.replace(dict) — целая строка ===
['груша', 'банан', 'cherry', 'apple pie'] 

=== B. цепочка str.replace(regex=False) — подстроки ===
['груша', 'банан', 'cherry', 'груша pie'] 

=== C. str.replace(dict) недоступен в этой версии ===
    TypeError : StringMethods.replace() missing 1 required positional argument: 'repl' 

=== cat(sep=' ') — NaN 'заражает' результат ===
['Иван Иванов', 'Пётр Сидоров', nan]
=== cat(sep=' ', na_rep='?') — NaN заменяется на '?' ===
['Иван Иванов', 'Пётр Сидоров', '? Смирнов'] 

=== pad(6, side='left', fillchar='0') — гибкий, но знак не учитывает ===
['000042', '000007', '000123', '0000-5']
=== zfill(6) — только нули, только слева, знак учитывает ===
['000042', '000007', '000123', '-000

Уровень 4. Архитектор

Задача 33 (Timestamp/Timedelta/Period). Создайте pd.Timestamp("2025-01-15 14:30") (время урока) — извлеките year, month, day, hour, dayofweek, day_name(), is_month_end. Прибавьте pd.Timedelta(days=5). Создайте pd.Timedelta("3 days 2 hours") — выведите total_seconds(). Создайте pd.Period("2025-01", freq="M") — выведите start_time, end_time, прибавьте 3. Примените pd.to_datetime(["2025-01-01", "invalid"], errors="coerce"). Ожидаемый вывод: Timestamp("2025-01-15").day_name() = "Wednesday"; Timedelta("3 days 2 hours").total_seconds() = 266400.0; Period("2025-01") + 3 = Period("2025-04", "M"); to_datetime(errors="coerce") даёт [Timestamp, NaT]. В комментариях объясните: чем Period отличается от Timestamp (период vs момент); почему errors="coerce" безопаснее raise (не ломает пайплайн); чем Timedelta отличается от Period (длительность vs календарный период); как Period обрабатывает границы месяца (start_time/end_time); что произойдёт при to_datetime с неверным форматом без errors (ValueError); как Timestamp работает с часовыми поясами (tz_localize/tz_convert).

In [17]:
# ============================================================
# Задача 33. Timestamp / Timedelta / Period
# ============================================================
import pandas as pd

# ------------------------------------------------------------
# 1. Timestamp — конкретный момент времени
# ------------------------------------------------------------
ts = pd.Timestamp("2025-01-15 14:30")
print("=== Timestamp('2025-01-15 14:30') ===")
print("year         =", ts.year)          # 2025
print("month        =", ts.month)         # 1
print("day          =", ts.day)           # 15
print("hour         =", ts.hour)          # 14
print("dayofweek    =", ts.dayofweek)     # 2  (пн=0 ... вс=6)
print("day_name()   =", ts.day_name())    # Wednesday
print("is_month_end =", ts.is_month_end)  # False
print("+5 дней      :", ts + pd.Timedelta(days=5), "\n")

# ------------------------------------------------------------
# 2. Timedelta — длительность
# ------------------------------------------------------------
td = pd.Timedelta("3 days 2 hours")
print("=== Timedelta('3 days 2 hours') ===")
print("total_seconds =", td.total_seconds())   # 266400.0
print("components    =", td.components, "\n")
# 3*86400 + 2*3600 = 259200 + 7200 = 266400

# ------------------------------------------------------------
# 3. Period — календарный интервал
# ------------------------------------------------------------
p = pd.Period("2025-01", freq="M")
print("=== Period('2025-01', freq='M') ===")
print("start_time =", p.start_time)   # 2025-01-01 00:00:00
print("end_time   =", p.end_time)     # 2025-01-31 23:59:59.999999999
print("+3 месяца  =", p + 3)          # Period('2025-04', 'M'), "\n"

# ------------------------------------------------------------
# 4. to_datetime с errors="coerce"
# ------------------------------------------------------------
print("=== to_datetime(errors='coerce') ===")
print(pd.to_datetime(["2025-01-01", "invalid"], errors="coerce").tolist(), "\n")

# ------------------------------------------------------------
# 5. Часовые пояса
# ------------------------------------------------------------
naive = pd.Timestamp("2025-01-15 14:30")
moscow = naive.tz_localize("Europe/Moscow")
print("=== Часовые пояса ===")
print("naive        :", naive)
print("tz_localize  :", moscow)
print("tz_convert   :", moscow.tz_convert("UTC"), "\n")

=== Timestamp('2025-01-15 14:30') ===
year         = 2025
month        = 1
day          = 15
hour         = 14
dayofweek    = 2
day_name()   = Wednesday
is_month_end = False
+5 дней      : 2025-01-20 14:30:00 

=== Timedelta('3 days 2 hours') ===
total_seconds = 266400.0
components    = Components(days=3, hours=2, minutes=0, seconds=0, milliseconds=0, microseconds=0, nanoseconds=0) 

=== Period('2025-01', freq='M') ===
start_time = 2025-01-01 00:00:00
end_time   = 2025-01-31 23:59:59.999999999
+3 месяца  = 2025-04
=== to_datetime(errors='coerce') ===
[Timestamp('2025-01-01 00:00:00'), NaT] 

=== Часовые пояса ===
naive        : 2025-01-15 14:30:00
tz_localize  : 2025-01-15 14:30:00+03:00
tz_convert   : 2025-01-15 11:30:00+00:00 



Задача 34 (date_range, DatetimeIndex, .dt). Создайте pd.date_range("2024-01-01", periods=365, freq="D"). Создайте диапазоны с freq="B", "ME", "YE", "h". Установите DatetimeIndex как индекс. Выполните df["2024"], df["2024-01"], df["2024-01-15"], df["2024-01-01":"2024-01-31"]. Примените .dt.year, .dt.month, .dt.day_name(), .dt.to_period("M"). Ожидаемый вывод: freq="B" даёт 260 рабочих дней в году; freq="ME" — конец каждого месяца; df["2024-01"] возвращает 31 строку; .dt.day_name() возвращает день недели. В комментариях объясните: почему freq="ME" вместо "M" в Pandas 3.0 (унификация с "YE", "QE"); как df["2024-01"] работает без .loc (частичная индексация по DatetimeIndex); когда .dt незаменим (извлечение компонент даты); чем частичная индексация отличается от .loc (более лаконичный синтаксис); что произойдёт при df["2024-13"] (KeyError или пустой результат); как date_range обрабатывает високосные годы (29 февраля включается при freq="D").

In [21]:
# ============================================================
# Задача 34. date_range, DatetimeIndex, .dt — рабочий вариант
# ============================================================
import pandas as pd

# --- 1. Диапазоны -------------------------------------------------
daily = pd.date_range("2024-01-01", periods=365, freq="D")
print("D: всего", len(daily),
      "| первый:", daily[0].date(),
      "| последний:", daily[-1].date())

full_year = pd.date_range("2024-01-01", "2024-12-31", freq="D")
print("D (полный год):", len(full_year), "| последний:", full_year[-1].date())

print("B:", pd.date_range("2024-01-01", periods=365, freq="B")[-1].date(),
      "  # 365 РАБОЧИХ дней ≈ 17 календарных месяцев")
print("ME:", pd.date_range("2024-01-01", periods=4, freq="ME").date.tolist())
print("YE:", pd.date_range("2024-01-01", periods=3, freq="YE").date.tolist())
print("h :", pd.date_range("2024-01-01", periods=3, freq="h").tolist(), "\n")

# --- 2. DataFrame с DatetimeIndex ---------------------------------
df = pd.DataFrame({"v": range(len(daily))}, index=daily)
assert isinstance(df.index, pd.DatetimeIndex), "индекс не DatetimeIndex!"
print("df.index.dtype =", df.index.dtype, "\n")

# --- 3. Частичная индексация ТОЛЬКО через .loc --------------------
# df["2024"]          -> ❌ KeyError: "2024" — это не столбец
# df.loc["2024"]      -> ✅ частичная индексация по DatetimeIndex

print("df.loc['2024'] len                =", len(df.loc["2024"]))       # 365
print("df.loc['2024-01'] len             =", len(df.loc["2024-01"]))    # 31
print("df.loc['2024-01-15']:\n",            df.loc["2024-01-15"], "\n") # Series (одна строка)
print("df.loc[['2024-01-15']]:\n",          df.loc[["2024-01-15"]], "\n")# DataFrame (одна строка)
print("df.loc['2024-01-01':'2024-01-31'] len =",
      len(df.loc["2024-01-01":"2024-01-31"]), "\n")                     # 31

# --- 4. .dt-компоненты --------------------------------------------
print(".dt.year      :", df.index[:3].year.tolist())
print(".dt.month     :", df.index[:3].month.tolist())
print(".dt.day_name() :", df.index[:3].day_name().tolist())
print(".dt.to_period :", df.index[:3].to_period("M").tolist(), "\n")

# --- 5. Невалидный месяц ------------------------------------------
try:
    df.loc["2024-13"]
except KeyError as e:
    print("df.loc['2024-13'] -> KeyError:", e)

D: всего 365 | первый: 2024-01-01 | последний: 2024-12-30
D (полный год): 366 | последний: 2024-12-31
B: 2025-05-23   # 365 РАБОЧИХ дней ≈ 17 календарных месяцев
ME: [datetime.date(2024, 1, 31), datetime.date(2024, 2, 29), datetime.date(2024, 3, 31), datetime.date(2024, 4, 30)]
YE: [datetime.date(2024, 12, 31), datetime.date(2025, 12, 31), datetime.date(2026, 12, 31)]
h : [Timestamp('2024-01-01 00:00:00'), Timestamp('2024-01-01 01:00:00'), Timestamp('2024-01-01 02:00:00')] 

df.index.dtype = datetime64[ns] 

df.loc['2024'] len                = 365
df.loc['2024-01'] len             = 31
df.loc['2024-01-15']:
 v    14
Name: 2024-01-15 00:00:00, dtype: int64 

df.loc[['2024-01-15']]:
              v
2024-01-15  14 

df.loc['2024-01-01':'2024-01-31'] len = 31 

.dt.year      : [2024, 2024, 2024]
.dt.month     : [1, 1, 1]
.dt.day_name() : ['Monday', 'Tuesday', 'Wednesday']
.dt.to_period : [Period('2024-01', 'M'), Period('2024-01', 'M'), Period('2024-01', 'M')] 

df.loc['2024-13'] -> KeyErro

Уровень 5. Эксперт

Задача 43 (np.where/select/clip). Примените np.where(scores < 2, 2, scores), np.select([scores < 3, scores < 4, scores >= 4], ["двоечник", "троечник", "хорошист"], default="?"), np.clip(scores, 1, 5). Сравните с scores.where(scores >= 2, 2) и scores.mask(scores > 4.5, 4.5). Примените np.log, np.sqrt, np.abs, np.round, np.cumsum, np.percentile. Ожидаемый вывод: np.where и .where дают одинаковый результат; np.select возвращает ["двоечник", "троечник", "хорошист"] по порядку; np.clip обрезает с обеих сторон; np.log(0) — -inf. В комментариях объясните: почему порядок условий в select важен (первое совпадение); когда clip лучше where (обрезать с двух сторон); чем np.where отличается от .where() (np.where возвращает NumPy-массив, .where() — Series); что вернёт np.log для 0 и отрицательных (-inf, nan); как np.cumsum связан с expanding().sum(); когда np.select предпочтительнее вложенного np.where (читаемость для 3+ условий).

In [23]:
# ============================================================
# Задача 43. np.where / np.select / np.clip
# ============================================================
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0. Исходные данные
# ------------------------------------------------------------
scores = pd.Series([1.5, 2.7, 3.2, 4.1, 4.8, 2.0, 3.9])
print("scores:", scores.tolist(), "\n")

# ------------------------------------------------------------
# 1. np.where — замена по одному условию
#    Там, где score < 2, ставим 2; иначе оставляем исходное.
# ------------------------------------------------------------
w = np.where(scores < 2, 2, scores)
print("=== np.where(scores < 2, 2, scores) ===")
print(w.tolist(), "| type:", type(w).__name__, "\n")
# ndarray: индекс Series теряется.

# ------------------------------------------------------------
# 2. np.select — несколько условий, первое совпадение
#    Порядок важен: 1.5 попадёт в «двоечник», а не в «троечник».
# ------------------------------------------------------------
labels = np.select(
    [scores < 3, scores < 4, scores >= 4],
    ["двоечник", "троечник", "хорошист"],
    default="?",
)
print("=== np.select — категории ===")
print(labels.tolist(), "\n")

# ------------------------------------------------------------
# 3. np.clip — обрезка с обеих сторон
# ------------------------------------------------------------
clipped = np.clip(scores, 1, 5)
print("=== np.clip(scores, 1, 5) ===")
print(clipped.tolist(), "\n")
# В наших данных нет значений <1 или >5 — результат без изменений.
# Для наглядности добавим крайние значения:
demo = pd.Series([0.3, 2.7, 4.1, 6.8])
print("demo:", demo.tolist(), "-> clip(1,5):",
      np.clip(demo, 1, 5).tolist(), "\n")

# ------------------------------------------------------------
# 4. Series.where и Series.mask
#    where(cond, other) — заменить, где cond ЛОЖНО
#    mask(cond, other)  — заменить, где cond ИСТИННО
# ------------------------------------------------------------
sw = scores.where(scores >= 2, 2)
print("=== scores.where(scores >= 2, 2) ===")
print(sw.tolist(), "| type:", type(sw).__name__, "\n")

sm = scores.mask(scores > 4.5, 4.5)
print("=== scores.mask(scores > 4.5, 4.5) ===")
print(sm.tolist(), "\n")

# ------------------------------------------------------------
# 5. Универсальные ufunc: log / sqrt / abs / round / cumsum / percentile
# ------------------------------------------------------------
with np.errstate(divide="ignore", invalid="ignore"):
    # log(0) -> -inf, log(-1) -> nan; подавляем RuntimeWarning
    log_res = np.log(np.array([1, 0, -1, np.e]))
print("=== np.log([1, 0, -1, e]) ===")
print(log_res.tolist(), "\n")

print("=== np.sqrt([4, 9, 16]) ===")
print(np.sqrt(np.array([4, 9, 16])).tolist(), "\n")

print("=== np.abs([-1, 2, -3]) ===")
print(np.abs(np.array([-1, 2, -3])).tolist(), "\n")

print("=== np.round(scores, 1) ===")
print(np.round(scores.values, 1).tolist(), "\n")

print("=== np.cumsum(scores) ===")
cum = np.cumsum(scores.values)
print(cum.tolist(), "\n")
# То же, что scores.expanding().sum():
print("expanding().sum():", scores.expanding().sum().tolist(), "\n")

print("=== np.percentile(scores, 50) ===")
print(np.percentile(scores, 50), "\n")

# ------------------------------------------------------------
# 6. Бонус: три способа обрезки снизу до 2
# ------------------------------------------------------------
print("=== три способа поднять минимум до 2 ===")
print("np.where     :", np.where(scores < 2, 2, scores).tolist())
print("Series.where :", scores.where(scores >= 2, 2).tolist())
print("Series.mask  :", scores.mask(scores < 2, 2).tolist())
# mask(cond, other) — заменить ТАМ, ГДЕ cond == True (условие «что заменить»).
# where(cond, other) — оставить где True, заменить где False (условие «что оставить»).

scores: [1.5, 2.7, 3.2, 4.1, 4.8, 2.0, 3.9] 

=== np.where(scores < 2, 2, scores) ===
[2.0, 2.7, 3.2, 4.1, 4.8, 2.0, 3.9] | type: ndarray 

=== np.select — категории ===
['двоечник', 'двоечник', 'троечник', 'хорошист', 'хорошист', 'двоечник', 'троечник'] 

=== np.clip(scores, 1, 5) ===
[1.5, 2.7, 3.2, 4.1, 4.8, 2.0, 3.9] 

demo: [0.3, 2.7, 4.1, 6.8] -> clip(1,5): [1.0, 2.7, 4.1, 5.0] 

=== scores.where(scores >= 2, 2) ===
[2.0, 2.7, 3.2, 4.1, 4.8, 2.0, 3.9] | type: Series 

=== scores.mask(scores > 4.5, 4.5) ===
[1.5, 2.7, 3.2, 4.1, 4.5, 2.0, 3.9] 

=== np.log([1, 0, -1, e]) ===
[0.0, -inf, nan, 1.0] 

=== np.sqrt([4, 9, 16]) ===
[2.0, 3.0, 4.0] 

=== np.abs([-1, 2, -3]) ===
[1, 2, 3] 

=== np.round(scores, 1) ===
[1.5, 2.7, 3.2, 4.1, 4.8, 2.0, 3.9] 

=== np.cumsum(scores) ===
[1.5, 4.2, 7.4, 11.5, 16.3, 18.3, 22.2] 

expanding().sum(): [1.5, 4.2, 7.4, 11.5, 16.3, 18.3, 22.2] 

=== np.percentile(scores, 50) ===
3.2 

=== три способа поднять минимум до 2 ===
np.where     : [2.0, 2.7, 3.

Задача 44 (broadcasting/axis/keepdims). Создайте scores 3×4 (классы × предметы) и offset = pd.Series([0.1, 0.2, 0.3, 0.4], index=scores.columns). Сложите scores + offset. Создайте offset_col = pd.Series([0.5, 1.0, 1.5], index=scores.index) — сложите через scores.add(offset_col, axis=0). Попробуйте scores + offset_col без axis. Примените scores.values - scores.values.mean(axis=1, keepdims=True) и без keepdims. Ожидаемый вывод: scores + offset выравнивает по столбцам; scores + offset_col без axis даёт NaN; keepdims=True даёт форму (3, 1), без — (3,) с ошибкой broadcasting. В комментариях объясните: почему (3,) несовместима с (3, 4) без axis (нужно выравнивание по строкам); как fill_value в .add() помогает; когда keepdims экономит код (сохранение размерности для broadcasting); что произойдёт при axis=2 (ValueError); как np.newaxis заменяет keepdims; как broadcasting работает с трёхмерными массивами.

In [25]:
# ============================================================
# Задача 44. broadcasting / axis / keepdims
# ============================================================
import numpy as np
import pandas as pd

# --- 1. scores 3×4 (классы × предметы) -----------------------
scores = pd.DataFrame(
    np.round(np.random.default_rng(0).uniform(2, 5, (3, 4)), 3),
    index=["5А", "5Б", "6А"],
    columns=["Матем", "Рус", "Физ", "Хим"],
)
print("scores:\n", scores, "\n")

# --- 2. Два вида смещений ------------------------------------
offset     = pd.Series([0.1, 0.2, 0.3, 0.4], index=scores.columns)  # по колонкам
offset_col = pd.Series([0.5, 1.0, 1.5],       index=scores.index)    # по строкам

# broadcast по столбцам: Series.index == DataFrame.columns
print("=== scores + offset (по столбцам) ===")
print((scores + offset).round(2), "\n")

# broadcast по строкам: явно указываем axis=0
print("=== scores.add(offset_col, axis=0) ===")
print(scores.add(offset_col, axis=0).round(2), "\n")

# --- 3. Без axis: Series выравнивается по СТОЛБЦАМ → NaN -----
print("=== scores + offset_col (без axis) → outer join, всё NaN ===")
print(scores + offset_col, "\n")
# offset_col.index = ["5А","5Б","6А"], scores.columns = ["Матем","Рус","Физ","Хим"]
# Общих меток нет → все значения NaN.
# Объединение индексов: столбцы расширились до {Матем,Рус,Физ,Хим,5А,5Б,6А} — 7 шт.

# --- 4. keepdims: (3,1) vs (3,) ------------------------------
arr = scores.values
mean_kd   = arr.mean(axis=1, keepdims=True)   # (3, 1)
mean_flat = arr.mean(axis=1)                  # (3,)
print("mean(axis=1, keepdims=True).shape =", mean_kd.shape)
print("mean(axis=1).shape                =", mean_flat.shape)

centered = arr - mean_kd      # (3,4) - (3,1) → broadcast OK
print("\narr - mean_keepdims (центрирование по строкам):\n", centered.round(2), "\n")

try:
    arr - mean_flat           # (3,4) - (3,) → ValueError
except ValueError as e:
    print("arr - mean(без keepdims) -> ValueError:", e, "\n")

# --- 5. fill_value: работает ТОЛЬКО DataFrame+DataFrame ------
# 5a) Series + axis + fill_value → NotImplementedError
try:
    scores.add(offset_col, axis=0, fill_value=0)
except NotImplementedError as e:
    print("=== Series + fill_value → NotImplementedError ===")
    print("   ", e, "\n")

# 5b) DataFrame + DataFrame + fill_value → ✅ работает
df_left  = scores[["Матем"]]                              # 3×1, все классы
df_right = pd.DataFrame({"Матем": [0.5, 1.0]},
                        index=["5А", "5Б"])               # 2×1, НЕТ "6А"
print("=== df_left.add(df_right, fill_value=0) ===")
print(df_left.add(df_right, fill_value=0).round(2), "\n")
# 5А: 3.911 + 0.5 ; 5Б: 4.440 + 1.0 ; 6А: 3.631 + 0(fill) = 3.631 (без изменений)
# Без fill_value 6А стал бы NaN.

# --- 6. Недопустимая ось -------------------------------------
try:
    scores.add(offset_col, axis=2)
except ValueError as e:
    print("=== axis=2 → ValueError ===")
    print("   ", e, "\n")

# --- 7. np.newaxis как замена keepdims -----------------------
manual = arr - arr.mean(axis=1)[:, np.newaxis]
print("arr - mean[:, np.newaxis] == arr - mean_keepdims:",
      np.allclose(manual, centered))

scores:
     Матем    Рус    Физ    Хим
5А  3.911  2.809  2.123  2.050
5Б  4.440  4.738  3.820  4.188
6А  3.631  4.805  4.448  2.008 

=== scores + offset (по столбцам) ===
    Матем   Рус   Физ   Хим
5А   4.01  3.01  2.42  2.45
5Б   4.54  4.94  4.12  4.59
6А   3.73  5.00  4.75  2.41 

=== scores.add(offset_col, axis=0) ===
    Матем   Рус   Физ   Хим
5А   4.41  3.31  2.62  2.55
5Б   5.44  5.74  4.82  5.19
6А   5.13  6.30  5.95  3.51 

=== scores + offset_col (без axis) → outer join, всё NaN ===
    5А  5Б  6А  Матем  Рус  Физ  Хим
5А NaN NaN NaN    NaN  NaN  NaN  NaN
5Б NaN NaN NaN    NaN  NaN  NaN  NaN
6А NaN NaN NaN    NaN  NaN  NaN  NaN 

mean(axis=1, keepdims=True).shape = (3, 1)
mean(axis=1).shape                = (3,)

arr - mean_keepdims (центрирование по строкам):
 [[ 1.19  0.09 -0.6  -0.67]
 [ 0.14  0.44 -0.48 -0.11]
 [-0.09  1.08  0.73 -1.72]] 

arr - mean(без keepdims) -> ValueError: operands could not be broadcast together with shapes (3,4) (3,)  

=== Series + fill_value 

Уровень 6. Гуру

Задача 53 (векторизация vs iterrows). Создайте DataFrame из 100 000 строк с score, weight. Реализуйте weighted = score * weight через iterrows(), itertuples(), apply(axis=1), векторизацию df["score"] * df["weight"]. Замерьте время через time.perf_counter() (прогрев + 5 замеров). Сравните np.where(df["score"] >= 4, "отличник", "обычный") с df["score"].apply(lambda x: ...). Ожидаемый вывод: iterrows ≈2.8 с, itertuples ≈0.09 с, apply ≈1.5 с, векторизация ≈0.001 с; ускорение векторизации относительно iterrows ≈2500×. В комментариях объясните: почему iterrows — антипаттерн (создаёт Series на каждую строку); когда itertuples уместен (когда нужны именованные кортежи); почему apply(axis=1) медленнее векторизации (Python-цикл); как np.where заменяет apply; что такое namedtuple (возвращает itertuples); как измерить время корректно (прогрев, несколько прогонов, perf_counter).

In [26]:
import numpy as np
import pandas as pd
import time

N = 100_000
rng = np.random.default_rng(0)
df = pd.DataFrame({
    "score":  rng.uniform(1, 5, N),
    "weight": rng.uniform(0.5, 2.0, N),
})

def bench(name, fn, runs=5):
    fn()  # прогрев
    t = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        t.append(time.perf_counter() - t0)
    print(f"{name:12s}: {min(t):.4f} s")

bench("iterrows",   lambda: np.array([r.score * r.weight for _, r in df.iterrows()]))
bench("itertuples", lambda: np.array([r.score * r.weight for r in df.itertuples()]))
bench("apply",      lambda: df.apply(lambda r: r["score"] * r["weight"], axis=1))
bench("vector",     lambda: df["score"] * df["weight"])

# np.where vs apply
t0 = time.perf_counter()
a = np.where(df["score"] >= 4, "отличник", "обычный")
t1 = time.perf_counter()
b = df["score"].apply(lambda x: "отличник" if x >= 4 else "обычный")
t2 = time.perf_counter()
print(f"\nnp.where: {t1-t0:.4f}s | apply: {t2-t1:.4f}s")

# === Комментарии ===
# - iterrows — антипаттерн: на каждую строку создаётся pd.Series (overhead
#   атрибутов, dtype-приведения). Медленно ~2500× относительно векторизации.
# - itertuples уместен, когда нужен быстрый питонический обход с именованными
#   полями: возвращает namedtuple (обращение r.score, r.weight без Series-overhead).
# - apply(axis=1) тоже Python-цикл по строкам — быстрее iterrows, но всё равно
#   в разы медленнее векторизации (сериализация строки в Series).
# - np.where заменяет apply по столбцу целиком: одна vectorized-операция,
#   сохраняет индекс pandas.Series, работает на всём массиве за раз.
# - namedtuple — это tuple с именованными полями; itertuples отдаёт именно их.
#   Обращение по имени быстрее словаря, памяти почти как у tuple.
# - Корректный бенчмарк: прогрев (JIT/кэш), 5 прогонов, берём min/median,
#   используем time.perf_counter() (монотонные высокоточные часы, не wall-clock).

iterrows    : 3.6137 s
itertuples  : 0.0657 s
apply       : 0.6696 s
vector      : 0.0002 s

np.where: 0.0026s | apply: 0.0215s


Задача 54 (chunksize/usecols/dtype). Создайте CSV на 500 000 строк (логи оценок). Прочитайте: целиком; через chunksize=100_000 с агрегацией суммы; через usecols=["grade_id", "score"]; через dtype={"grade_id": "int32", "score": "float32"}. Сравните время и память (memory_usage(deep=True)). Ожидаемый вывод: chunksize даёт пиковую память ≈1/5 полного чтения; usecols сокращает память пропорционально числу столбцов; dtype уменьшает память в 2 раза для чисел. В комментариях объясните: когда chunksize обязателен (файл не влезает в память); как usecols экономит память; почему dtype при чтении лучше astype после (не создаётся промежуточный object); как агрегировать по чанкам (суммировать результаты); что произойдёт при chunksize больше размера файла (один чанк); как nrows помогает при тестировании (быстрая проверка структуры).

In [28]:
import numpy as np
import pandas as pd
import time, os, tracemalloc

N = 500_000
rng = np.random.default_rng(0)
path = "grades.csv"

pd.DataFrame({
    "grade_id":   rng.integers(1, 10_000, N),
    "student_id": rng.integers(1, 1_000, N),
    "subject_id": rng.integers(1, 10, N),
    "score":      rng.uniform(1, 5, N),
    "comment":    rng.choice(["ok", "bad", "good", "?"], N),
}).to_csv(path, index=False)

def time_read(**kw):
    """Чистое время БЕЗ tracemalloc."""
    t0 = time.perf_counter()
    df = pd.read_csv(path, **kw)
    return df, time.perf_counter() - t0

def peak_read(**kw):
    """Пик памяти через tracemalloc с явным reset."""
    tracemalloc.start()
    tracemalloc.reset_peak()          # ← ключевое: гарантированный сброс
    df = pd.read_csv(path, **kw)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return df, peak / 1e6

# 1) full
df_full, t_full = time_read()
m_full = df_full.memory_usage(deep=True).sum() / 1e6
print(f"full     : {t_full:.3f}s, usage {m_full:.1f} MB")

# 2) usecols
df_uc, t_uc = time_read(usecols=["grade_id", "score"])
m_uc = df_uc.memory_usage(deep=True).sum() / 1e6
print(f"usecols  : {t_uc:.3f}s, usage {m_uc:.1f} MB")

# 3) dtype для ВСЕХ колонок
all_dtypes = {"grade_id": "int32", "student_id": "int32",
              "subject_id": "int8", "score": "float32",
              "comment": "string"}
df_dt, t_dt = time_read(dtype=all_dtypes)
m_dt = df_dt.memory_usage(deep=True).sum() / 1e6
print(f"dtype-all: {t_dt:.3f}s, usage {m_dt:.1f} MB")

# 4) dtype только числовых — честное сравнение по числам
numeric_cols = ["grade_id", "student_id", "subject_id", "score"]
m_full_num = df_full[numeric_cols].memory_usage(deep=True).sum() / 1e6
df_dt2, t_dt2 = time_read(dtype={"grade_id": "int32", "student_id": "int32",
                                  "subject_id": "int8", "score": "float32"})
m_dt2_num = df_dt2[numeric_cols].memory_usage(deep=True).sum() / 1e6
print(f"\nчисловые колонки — full: {m_full_num:.1f} MB | dtype: {m_dt2_num:.1f} MB "
      f"| экономия {m_full_num/m_dt2_num:.1f}×")

# 5) chunksize — отдельный замер пика через tracemalloc
tracemalloc.start()
tracemalloc.reset_peak()
total = 0.0
for chunk in pd.read_csv(path, chunksize=100_000):
    total += chunk["score"].sum()
_, peak_chunk = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"chunks   : peak {peak_chunk/1e6:.1f} MB, sum={total:.1f}")

os.remove(path)

full     : 0.270s, usage 41.7 MB
usecols  : 0.171s, usage 8.0 MB
dtype-all: 0.334s, usage 32.2 MB

числовые колонки — full: 16.0 MB | dtype: 6.5 MB | экономия 2.5×
chunks   : peak 13.0 MB, sum=1500099.7
